In [ ]:
!uv pip install ipynbname
import ipynbname

__file__ = ipynbname.path()

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(__file__).parent.parent
FIGURE_DIR = Path(__file__).parent / 'figures'
LABELING_BASE = BASE / 'labeling/categorization/labels/final'
LABEL_FILE = {
    'claude-code': 'claude-code.csv',
    'codex': 'codex.csv',
    'devin': 'devin.csv',
    'human': 'human_labels.csv',
    'pr-agent': 'pr-agent.csv',
}

def get_labeling_data(tool):
    label_file = LABEL_FILE[tool]
    label_path = LABELING_BASE / label_file
    df = pd.read_csv(label_path)
    df = df[df['resolved_label'].notna()]
    df = df[['review_id', 'instance_id', 'comment_index', 'resolved_label']]
    return df

def plot_label_distribution(df, tool):
    import matplotlib.pyplot as plt
    from matplotlib import ticker

    label_percentage = (
        df['resolved_label']
        .value_counts(normalize=True)
        .mul(100)
        .sort_values(ascending=True)
    )
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#f8fafc')

    cmap = plt.get_cmap('YlGnBu')
    colors = [
        cmap(0.35 + 0.45 * idx / max(len(label_percentage) - 1, 1))
        for idx in range(len(label_percentage))
    ]
    bars = ax.barh(
        label_percentage.index,
        label_percentage.values,
        color=colors,
        edgecolor='#0f172a',
        linewidth=0.6,
        height=0.72,
    )

    tool_title = tool.replace('-', ' ').title()
    ax.set_title(
        f'Resolved Label Distribution for {tool_title}',
        fontsize=16,
        fontweight='bold',
        loc='left',
        pad=14,
    )
    ax.text(
        0,
        1.02,
        f'{len(df):,} resolved review comments',
        transform=ax.transAxes,
        fontsize=10,
        color='#475569',
    )
    ax.set_xlabel('Share of resolved labels (%)', labelpad=10)
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_xlim(0, label_percentage.max() * 1.18)
    ax.grid(axis='x', linestyle='--', linewidth=0.8, alpha=0.35)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#94a3b8')
    ax.tick_params(axis='y', length=0, pad=8)
    ax.tick_params(axis='x', colors='#475569')

    for bar, value in zip(bars, label_percentage.values):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f'{value:.1f}%',
            va='center',
            ha='left',
            fontsize=10,
            color='#0f172a',
        )

    tool_slug = tool.replace('-', '_')
    figure_stem = f'resolved_label_distribution_{tool_slug}'
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    svg_path = FIGURE_DIR / f'{figure_stem}.svg'
    pdf_path = FIGURE_DIR / f'{figure_stem}.pdf'
    fig.savefig(svg_path, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    print(f'Saved {svg_path.relative_to(BASE)}')
    print(f'Saved {pdf_path.relative_to(BASE)}')
    plt.show()

def get_label_percentages(tool, category_order=None):
    label_percentage = (
        get_labeling_data(tool)['resolved_label']
        .value_counts(normalize=True)
        .mul(100)
    )
    if category_order is not None:
        label_percentage = label_percentage.reindex(category_order, fill_value=0)
    return label_percentage

def plot_tool_grid_with_human_overlay(tools=('pr-agent', 'devin', 'codex', 'claude-code'), reference_tool='human', layout='2x2'):
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib import ticker
    from matplotlib.patches import Patch

    tool_colors = {
        'pr-agent': '#2563eb',
        'devin': '#0f766e',
        'codex': '#c2410c',
        'claude-code': '#b91c1c',
    }
    tool_titles = {
        'pr-agent': 'PR-Agent',
        'devin': 'Devin',
        'codex': 'Codex',
        'claude-code': 'Claude Code',
    }
    human_fill = '#f8f1dd'
    human_hatch = '#e5cf9b'
    human_marker_fill = '#f2c35b'
    tools = tuple(tools)

    human_percentage = get_label_percentages(reference_tool)
    category_order = human_percentage.sort_values(ascending=False).index.tolist()
    human_percentage = human_percentage.reindex(category_order)
    tool_percentages = {
        tool: get_label_percentages(tool, category_order)
        for tool in tools
    }
    if layout == '2x2':
        nrows, ncols = 2, 2
        figsize = (7.15, 4.45)
        gridspec_kw = {'wspace': 0.10, 'hspace': 0.10}
        legend_y = 1.1
    elif layout == '1x4':
        nrows, ncols = 1, 4
        figsize = (7.15, 2.45)
        gridspec_kw = {'wspace': 0.10, 'hspace': 0.0}
        legend_y = 1.1
    else:
        raise ValueError("layout must be '2x2' or '1x4'")

    y_positions = np.arange(len(category_order))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        sharex=False,
        sharey=True,
        constrained_layout=True,
        gridspec_kw=gridspec_kw,
    )
    fig.patch.set_facecolor('none')
    fig.patch.set_alpha(0)
    axes = np.atleast_1d(axes).ravel()

    legend_handles = [
        Patch(facecolor='#64748b', edgecolor='none', alpha=0.9, label='Tool Reviews'),
        Patch(
            facecolor=human_fill,
            edgecolor=human_hatch,
            hatch='////',
            linewidth=0.8,
            alpha=0.82,
            label='Human Reviews',
        ),
    ]
    fig.legend(
        handles=legend_handles,
        loc='upper center',
        bbox_to_anchor=(0.5, legend_y),
        ncol=2,
        frameon=False,
        fontsize=7.4,
        handlelength=1.6,
        columnspacing=1.2,
    )

    for panel_index, (ax, tool) in enumerate(zip(axes, tools)):
        _, col_index = divmod(panel_index, ncols)
        is_left_column = col_index == 0
        distribution = tool_percentages[tool]
        panel_max = max(distribution.max(), human_percentage.max())
        panel_limit = int(np.ceil(max(12, panel_max + 4.0) / 5) * 5)
        tick_step = 10 if panel_limit > 25 else 5
        if tool == 'codex' and panel_limit >= 40:
            tick_step = 20
        bars = ax.barh(
            y_positions,
            distribution.values,
            height=0.50,
            color=tool_colors.get(tool, '#2563eb'),
            edgecolor='none',
            alpha=0.92,
            zorder=2,
        )
        ax.barh(
            y_positions,
            human_percentage.values,
            height=0.20,
            color=human_fill,
            edgecolor='none',
            alpha=0.78,
            zorder=3,
        )
        ax.barh(
            y_positions,
            human_percentage.values,
            height=0.20,
            color='none',
            edgecolor=human_hatch,
            linewidth=0.75,
            hatch='////',
            zorder=4,
        )
        ax.scatter(
            human_percentage.values,
            y_positions,
            marker='none',
            s=12,
            color=human_marker_fill,
            linewidth=0.7,
            zorder=5,
        )

        ax.set_facecolor('none')
        ax.set_title(
            tool_titles[tool],
            fontsize=8.9,
            fontweight='bold',
            loc='left',
            pad=2,
        )
        ax.set_xlim(0, panel_limit)
        ax.margins(y=0.05)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(tick_step))
        ax.xaxis.set_major_formatter(
            ticker.FuncFormatter(
                lambda value, pos: f'{int(value)}%'
            )
        )
        ax.grid(axis='x', linestyle=':', linewidth=0.65, alpha=0.35)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color('#94a3b8')
        ax.tick_params(axis='x', colors='#475569', labelsize=7.0, pad=1.2)

        if is_left_column:
            ax.set_yticks(y_positions, category_order)
            ax.tick_params(axis='y', labelleft=True, length=0, pad=1.6, labelsize=6.8)
        else:
            ax.tick_params(axis='y', labelleft=False, length=0)

        for bar, label, value in zip(bars, category_order, distribution.values):
            delta_value = value - human_percentage.loc[label]
            if round(abs(delta_value), 1) < 3.2:
                continue

            human_value = human_percentage.loc[label]
            text_x = max(value, human_value) + 0.35
            if abs(text_x - human_value) < 0.65:
                text_x += 0.45
            delta_color = '#15803d' if delta_value > 0 else '#b91c1c'

            ax.text(
                text_x,
                bar.get_y() + bar.get_height() / 2,
                f'{delta_value:+.1f} pp',
                va='center',
                ha='left',
                fontsize=6.2,
                fontweight='bold',
                color=delta_color,
                clip_on=False,
                zorder=5,
            )

    for ax in axes:
        ax.invert_yaxis()

    fig.supxlabel('Share of categories (%)', fontsize=8.0, y=-0.05)

    figure_stem = f'rq2_categorization_{layout}'
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    pdf_path = FIGURE_DIR / f'{figure_stem}.pdf'
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True, pad_inches=0.01)
    print(f'Saved {pdf_path.relative_to(BASE)}')
    plt.show()

EVAL_RESULTS_BY_COMMENT_TEXT = BASE / 'results_eval_combined_resolved_true_by_comment_text'

def get_resolved_label_data(tool):
    label_file = LABEL_FILE[tool]
    label_path = LABELING_BASE / label_file
    df = pd.read_csv(label_path)
    return df[df['resolved_label'].notna()].copy()

def get_tool_eval_results(tool):
    import json

    rows = []
    tool_dir = EVAL_RESULTS_BY_COMMENT_TEXT / tool
    for result_path in sorted(tool_dir.glob('*/result.json')):
        payload = json.loads(result_path.read_text())
        instance_id = payload.get('instance_id') or result_path.parent.name
        for item in payload.get('results', []):
            rows.append(
                {
                    'tool': tool,
                    'instance_id': instance_id,
                    'comment_index': item.get('comment_index'),
                    'comment_text': item.get('comment_text'),
                    'test_passed': item.get('test_passed'),
                }
            )
    return pd.DataFrame(rows)

def get_pass_rate_by_human_category(tool, reference_tool='human'):
    import difflib
    import re
    import unicodedata

    human_df = get_resolved_label_data(reference_tool)[
        ['instance_id', 'comment_index', 'comment_text', 'resolved_label']
    ].copy()

    def normalize_comment_text(text):
        text = unicodedata.normalize('NFKC', str(text or ''))
        for old in ('\\r', '\\n', '\r', '\n', '\xa0'):
            text = text.replace(old, ' ')
        text = text.replace('\ufffd', ' ')
        text = text.encode('ascii', 'ignore').decode()
        text = re.sub(r'_{2,}', ' ', text)
        text = re.sub(r'(?<![0-9A-Za-z])_(?![0-9A-Za-z])', ' ', text)
        text = re.sub(r'(?<![0-9A-Za-z])_(?=[a-z])', ' ', text)
        text = re.sub(r'(?<=[a-z])_(?![0-9A-Za-z])', ' ', text)
        text = re.sub(r'\\{2,}', r'\\', text)
        text = re.sub(r'\.{3,}$', '', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    human_df['normalized_comment_text'] = human_df['comment_text'].map(normalize_comment_text)
    category_order = human_df['resolved_label'].value_counts().index.tolist()

    human_by_text = human_df.drop_duplicates(['instance_id', 'normalized_comment_text'])[
        ['instance_id', 'normalized_comment_text', 'resolved_label']
    ].copy()
    human_instance_ids = set(human_by_text['instance_id'])
    human_candidates = {
        instance_id: group[['normalized_comment_text', 'resolved_label']]
        .drop_duplicates()
        .to_dict('records')
        for instance_id, group in human_by_text.groupby('instance_id')
    }

    eval_df = get_tool_eval_results(tool).copy()
    eval_df['normalized_comment_text'] = eval_df['comment_text'].map(normalize_comment_text)
    matched_exact = eval_df.merge(
        human_by_text,
        on=['instance_id', 'normalized_comment_text'],
        how='inner',
    ).copy()
    matched_exact['match_source'] = 'comment_text_exact'

    unmatched_eval = eval_df.merge(
        matched_exact[['instance_id', 'normalized_comment_text']],
        on=['instance_id', 'normalized_comment_text'],
        how='left',
        indicator=True,
    )
    unmatched_eval = unmatched_eval[unmatched_eval['_merge'] == 'left_only'].drop(columns=['_merge'])

    fuzzy_rows = []
    for row in unmatched_eval.itertuples(index=False):
        candidates = human_candidates.get(row.instance_id, [])
        if not candidates:
            continue
        scores = sorted(
            (
                difflib.SequenceMatcher(
                    None,
                    row.normalized_comment_text,
                    candidate['normalized_comment_text'],
                ).ratio(),
                candidate,
            )
            for candidate in candidates
        )
        best_score, best_candidate = scores[-1]
        second_best = scores[-2][0] if len(scores) > 1 else 0.0
        if best_score >= 0.97 and (best_score - second_best) >= 0.03:
            fuzzy_rows.append(
                {
                    **row._asdict(),
                    'resolved_label': best_candidate['resolved_label'],
                    'match_source': 'comment_text_fuzzy',
                }
            )

    matched = pd.concat(
        [matched_exact, pd.DataFrame(fuzzy_rows)],
        ignore_index=True,
        sort=False,
    )
    matched['test_passed'] = matched['test_passed'].fillna(False).astype(bool)

    summary = (
        matched.groupby('resolved_label')
        .agg(
            pass_rate=('test_passed', 'mean'),
            num_passed=('test_passed', 'sum'),
            n=('test_passed', 'size'),
        )
        .reindex(category_order, fill_value=0)
        .reset_index()
        .rename(columns={'resolved_label': 'category'})
    )
    summary['pass_rate'] = summary['pass_rate'] * 100
    summary['num_passed'] = summary['num_passed'].astype(int)

    stats = {
        'matched_comments': int(len(matched)),
        'total_eval_comments': int(len(eval_df)),
        'total_human_comments': int(len(human_df)),
        'shared_instance_eval_comments': int(eval_df['instance_id'].isin(human_instance_ids).sum()),
        'matched_instances': int(matched['instance_id'].nunique()),
        'join_by_text_exact': int((matched['match_source'] == 'comment_text_exact').sum()),
        'join_by_text_fuzzy': int((matched['match_source'] == 'comment_text_fuzzy').sum()),
        'overall_pass_rate': float(matched['test_passed'].mean() * 100) if len(matched) else 0.0,
    }
    return summary, matched, stats

def plot_tool_pass_rate_by_human_category(
    tools=('pr-agent', 'devin', 'codex', 'claude-code'),
    reference_tool='human',
    layout='1x4',
):
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib import ticker

    tool_colors = {
        'pr-agent': '#2563eb',
        'devin': '#0f766e',
        'codex': '#c2410c',
        'claude-code': '#b91c1c',
    }
    tool_titles = {
        'pr-agent': 'PR-Agent',
        'devin': 'Devin',
        'codex': 'Codex',
        'claude-code': 'Claude Code',
    }
    tools = tuple(tools)

    human_df = get_resolved_label_data(reference_tool)
    category_order = human_df['resolved_label'].value_counts().index.tolist()
    tool_summaries = {}
    tool_stats = {}
    max_pass_rate = 0.0

    for tool in tools:
        summary, _, stats = get_pass_rate_by_human_category(tool, reference_tool=reference_tool)
        summary = summary.set_index('category').reindex(category_order, fill_value=0).reset_index()
        tool_summaries[tool] = summary
        tool_stats[tool] = stats
        max_pass_rate = max(max_pass_rate, float(summary['pass_rate'].max()))

    if layout == '2x2':
        nrows, ncols = 2, 2
        figsize = (7.15, 4.45)
        gridspec_kw = {'wspace': 0.10, 'hspace': 0.12}
    elif layout == '1x4':
        nrows, ncols = 1, 4
        figsize = (7.15, 2.65)
        gridspec_kw = {'wspace': 0.10, 'hspace': 0.0}
    else:
        raise ValueError("layout must be '2x2' or '1x4'")

    x_limit = max(40, int(np.ceil((max_pass_rate + 12) / 5) * 5))
    tick_step = 20 if x_limit >= 60 else 10
    y_positions = np.arange(len(category_order))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        sharex=True,
        sharey=True,
        constrained_layout=True,
        gridspec_kw=gridspec_kw,
    )
    fig.patch.set_facecolor('none')
    fig.patch.set_alpha(0)
    axes = np.atleast_1d(axes).ravel()

    for panel_index, (ax, tool) in enumerate(zip(axes, tools)):
        _, col_index = divmod(panel_index, ncols)
        is_left_column = col_index == 0
        summary = tool_summaries[tool]
        stats = tool_stats[tool]

        bars = ax.barh(
            y_positions,
            summary['pass_rate'].to_numpy(),
            height=0.54,
            color=tool_colors[tool],
            edgecolor='none',
            alpha=0.92,
            zorder=2,
        )

        ax.set_facecolor('none')
        ax.axvline(
            stats['overall_pass_rate'],
            color='#f9a8d4',
            linestyle=':',
            linewidth=1.2,
            alpha=0.68,
            zorder=1,
        )
        ax.set_title(
            tool_titles[tool],
            fontsize=8.9,
            fontweight='bold',
            loc='left',
            pad=2,
        )
        ax.set_xlim(0, x_limit)
        ax.margins(y=0.05)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(tick_step))
        ax.xaxis.set_major_formatter(
            ticker.FuncFormatter(lambda value, pos: f'{int(value)}%')
        )
        ax.grid(axis='x', linestyle=':', linewidth=0.65, alpha=0.35)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color('#94a3b8')
        ax.tick_params(axis='x', colors='#475569', labelsize=7.0, pad=1.2)

        if is_left_column:
            ax.set_yticks(y_positions, category_order)
            ax.tick_params(axis='y', labelleft=True, length=0, pad=1.6, labelsize=6.8)
        else:
            ax.tick_params(axis='y', labelleft=False, length=0)

        for bar, value, num_passed, total_count in zip(
            bars,
            summary['pass_rate'].to_numpy(),
            summary['num_passed'].to_numpy(),
            summary['n'].to_numpy(),
        ):
            text_x = min(value + 0.9, x_limit - 0.5)
            ax.text(
                text_x,
                bar.get_y() + bar.get_height() / 2,
                f'{value:.1f}% ({num_passed}/{int(total_count)})',
                va='center',
                ha='left',
                fontsize=6.2,
                color='#0f172a',
                clip_on=False,
                zorder=3,
            )

    for ax in axes:
        ax.invert_yaxis()

    fig.supxlabel('Average test pass rate (%)', fontsize=8.0, y=-0.05)

    figure_stem = f'rq2_pass_rate_by_human_category_{layout}'
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    pdf_path = FIGURE_DIR / f'{figure_stem}.pdf'
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True, pad_inches=0.01)

    for tool in tools:
        stats = tool_stats[tool]
        print(
            f"{tool}: matched {stats['matched_comments']}/{stats['total_eval_comments']} result comments "
            f"({stats['join_by_text_exact']} exact text, {stats['join_by_text_fuzzy']} high-similarity text); "
            f"resolved human CSV has {stats['total_human_comments']} comments"
        )
    print(f'Saved {pdf_path.relative_to(BASE)}')
    plt.show()


In [ ]:
plot_tool_grid_with_human_overlay(['pr-agent', 'devin', 'codex', 'claude-code'], layout='1x4')

In [ ]:
plot_tool_pass_rate_by_human_category(['pr-agent', 'devin', 'codex', 'claude-code'], layout='1x4')